# Re-evaluate Baseline Outputs To All Results

Notebook ini tidak menjalankan inference/model. Ia hanya membaca output WAV yang sudah ada di `outputs/<model>/manifest.csv`, menghitung metrik core, lalu menulis ulang `<model>_results.csv` dan `all_results.csv`.

In [ ]:
from __future__ import annotations

import importlib.util
import os
from pathlib import Path
import subprocess
import sys
from datetime import datetime


PIPELINE_STAGE_NAME = "code_v4_musicnet_cqtdiffplus_44k"
COLAB_DRIVE_ROOT = Path(os.environ.get("COLAB_DRIVE_ROOT", "/content/drive/MyDrive"))
DEFAULT_DRIVE_DATA_ROOT = COLAB_DRIVE_ROOT / "THESIS CODE"

MODEL_NAME = os.environ.get("RUN_MODELS", "baseline_cqtdiff_finetuned").split(",")[0].strip()
TARGET_SR = int(os.environ.get("PIPELINE_TARGET_SR", "44100"))
SEGMENT_SAMPLES = int(os.environ.get("PIPELINE_SEGMENT_SAMPLES", "184184"))
SEGMENT_DURATION = SEGMENT_SAMPLES / TARGET_SR
EXPERIMENT_CONFIG_ID = (
    f"musicnet_cqtdiffplus_sr{TARGET_SR}_n{SEGMENT_SAMPLES}_"
    f"dur{SEGMENT_DURATION:.6f}s"
)
GAP_DURATIONS_MS = [100, 300, 500, 750, 1200, 1700]
DATASET_RANDOM_SEED = 42
EVAL_GAP_POSITION = os.environ.get("EVAL_GAP_POSITION", "center").strip().lower()
EVAL_RANDOM_GAP_MIN_CONTEXT_MS = int(os.environ.get("EVAL_RANDOM_GAP_MIN_CONTEXT_MS", "250"))
REPAIR_NUMPY_STACK = os.environ.get("REPAIR_NUMPY_STACK", "1").lower() in {"1", "true", "yes", "on"}


def info(message: str) -> None:
    print(f"[outputs-reeval] {message}", flush=True)


def fail(message: str) -> None:
    raise SystemExit(f"\nERROR: {message}\n")


In [ ]:
def run_command(command: list[str], cwd: Path | str | None = None) -> None:
    location = f" (cwd={cwd})" if cwd is not None else ""
    info("Running: " + " ".join(map(str, command)) + location)
    subprocess.run([str(part) for part in command], check=True, cwd=str(cwd) if cwd is not None else None)


def ensure_numeric_stack() -> None:
    try:
        import numpy as _np
        _ = _np.random.default_rng(0).normal(size=1)
        import pandas as _pd
        import scipy as _scipy
        import soundfile as _sf
        import librosa as _librosa
        info(
            "Numeric/audio stack OK: "
            f"numpy={_np.__version__}, pandas={_pd.__version__}, scipy={_scipy.__version__}, "
            f"librosa={_librosa.__version__}"
        )
        return
    except Exception as exc:
        if not REPAIR_NUMPY_STACK:
            fail(f"Import numeric/audio stack gagal: {exc}")
        info(f"Numeric/audio stack bermasalah: {exc}")
        info("Reinstall stack yang kompatibel dengan numpy==1.26.4.")
        run_command([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-cache-dir",
            "numpy==1.26.4",
            "pandas==2.2.2",
            "scipy==1.13.1",
            "scikit-learn==1.5.2",
            "librosa==0.10.2.post1",
            "soundfile==0.12.1",
        ])
        raise SystemExit(
            "Dependency binary stack sudah direinstall. Restart runtime/kernel dulu, "
            "lalu run notebook ini lagi dari awal."
        )


ensure_numeric_stack()

import numpy as np
import pandas as pd
import soundfile as sf
import librosa



In [ ]:
def is_colab_runtime() -> bool:
    return (
        "COLAB_GPU" in os.environ
        or "google.colab" in sys.modules
        or importlib.util.find_spec("google.colab") is not None
    )


def maybe_mount_google_drive() -> None:
    if not is_colab_runtime():
        return
    if COLAB_DRIVE_ROOT.exists():
        info(f"Google Drive already mounted: {COLAB_DRIVE_ROOT}")
        return
    from google.colab import drive
    info("Mounting Google Drive")
    drive.mount("/content/drive")


def project_root() -> Path:
    if "PROJECT_ROOT" in os.environ:
        return Path(os.environ["PROJECT_ROOT"]).resolve()
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path.cwd().resolve()


def data_root() -> Path:
    if "MUSIC_INPAINTING_ROOT" in os.environ:
        return Path(os.environ["MUSIC_INPAINTING_ROOT"]).resolve()
    if is_colab_runtime():
        return DEFAULT_DRIVE_DATA_ROOT.resolve()
    return (project_root() / "music_inpainting").resolve()


def paths() -> dict[str, Path]:
    base = data_root()
    stage = base / "training_stages" / PIPELINE_STAGE_NAME
    return {
        "base": base,
        "stage": stage,
        "preprocessed": Path(os.environ.get("BASELINE_EVAL_PREPROCESSED_DIR", stage / "preprocessed")).resolve(),
        "outputs": (stage / "outputs").resolve(),
        "results": (stage / "results").resolve(),
    }


def evaluation_artifact_name(model_name: str) -> str:
    if EVAL_GAP_POSITION == "center":
        return model_name
    return f"{model_name}_{EVAL_GAP_POSITION}gap"


In [ ]:
def compute_gap_bounds(audio_length: int, gap_ms: int, sr: int = TARGET_SR) -> tuple[int, int]:
    gap_samples = int(round(sr * gap_ms / 1000))
    center = int(audio_length) // 2
    gap_start = center - gap_samples // 2
    return int(gap_start), int(gap_start + gap_samples)


def build_gap_mask_array(audio_length: int, gap_ms: int, sr: int = TARGET_SR,
                         gap_start: int | None = None) -> tuple[np.ndarray, int, int]:
    gap_samples = int(round(sr * gap_ms / 1000))
    if gap_start is None:
        gap_start, gap_end = compute_gap_bounds(audio_length, gap_ms, sr=sr)
    else:
        gap_start = int(gap_start)
        gap_end = gap_start + gap_samples
    mask = np.zeros(audio_length, dtype=bool)
    mask[gap_start:gap_end] = True
    return mask, int(gap_start), int(gap_end)


def make_eval_gap_mask(audio_length: int, gap_ms: int, sample_index: int,
                       sr: int = TARGET_SR) -> tuple[np.ndarray, int, int]:
    if EVAL_GAP_POSITION == "center":
        return build_gap_mask_array(audio_length, gap_ms, sr=sr)
    gap_samples = int(round(sr * gap_ms / 1000))
    min_context = int(round(sr * EVAL_RANDOM_GAP_MIN_CONTEXT_MS / 1000))
    min_start = min_context
    max_start = audio_length - gap_samples - min_context
    if max_start < min_start:
        min_start = 0
        max_start = audio_length - gap_samples
    if max_start < min_start:
        raise ValueError(f"Random gap {gap_ms}ms tidak valid untuk audio_length={audio_length}.")
    rng = np.random.default_rng(DATASET_RANDOM_SEED + 100_000 + int(sample_index) * 997 + int(gap_ms))
    return build_gap_mask_array(audio_length, gap_ms, sr=sr, gap_start=int(rng.integers(min_start, max_start + 1)))


def _stratified_sample_table(df: pd.DataFrame, n_samples: int, seed: int = DATASET_RANDOM_SEED,
                             stratify_cols=("composer", "instrument")) -> pd.DataFrame:
    if len(df) == 0 or n_samples <= 0:
        return df.iloc[0:0].copy()
    work = df.copy().reset_index(drop=True)
    work["_sample_row_id"] = np.arange(len(work))
    n_samples = min(int(n_samples), len(work))
    for col in stratify_cols:
        if col not in work.columns:
            work[col] = "unknown"
        work[col] = work[col].fillna("unknown").astype(str)
    rng = np.random.default_rng(seed)
    grouped = list(work.groupby(list(stratify_cols), dropna=False, sort=True))
    quotas = []
    for _, group in grouped:
        expected = len(group) * n_samples / len(work)
        base = int(np.floor(expected))
        quotas.append({"group": group, "quota": min(base, len(group)), "fractional": expected - base, "tie": rng.random()})
    remaining = n_samples - sum(q["quota"] for q in quotas)
    for q in sorted(quotas, key=lambda item: (-item["fractional"], item["tie"])):
        if remaining <= 0:
            break
        capacity = len(q["group"]) - q["quota"]
        if capacity > 0:
            q["quota"] += 1
            remaining -= 1
    parts = []
    for offset, q in enumerate(quotas):
        if q["quota"] > 0:
            parts.append(q["group"].sample(q["quota"], random_state=seed + offset))
    sampled = pd.concat(parts, ignore_index=True) if parts else work.iloc[0:0].copy()
    if len(sampled) < n_samples:
        missing = n_samples - len(sampled)
        sampled_ids = set(sampled["_sample_row_id"]) if "_sample_row_id" in sampled.columns else set()
        unsampled = work[~work["_sample_row_id"].isin(sampled_ids)]
        if len(unsampled) > 0:
            sampled = pd.concat([sampled, unsampled.sample(min(missing, len(unsampled)), random_state=seed + 999)], ignore_index=True)
    return sampled.sample(frac=1.0, random_state=seed).drop(columns=["_sample_row_id"], errors="ignore").reset_index(drop=True)


def get_data_splits(meta_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    source_cols = ["source_file"]
    for col in ["composer", "instrument"]:
        if col in meta_df.columns:
            source_cols.append(col)
    source_df = meta_df[source_cols].drop_duplicates("source_file").reset_index(drop=True)
    for col in ["composer", "instrument"]:
        if col not in source_df.columns:
            source_df[col] = "unknown"
        source_df[col] = source_df[col].fillna("unknown").astype(str)
    n_test = max(1, int(round(len(source_df) * 0.15)))
    n_val = max(1, int(round(len(source_df) * 0.15)))
    test_sources_df = _stratified_sample_table(source_df, n_test, seed=DATASET_RANDOM_SEED)
    remaining_sources = source_df[~source_df["source_file"].isin(test_sources_df["source_file"])].reset_index(drop=True)
    val_sources_df = _stratified_sample_table(remaining_sources, n_val, seed=DATASET_RANDOM_SEED + 1)
    train_sources_df = remaining_sources[~remaining_sources["source_file"].isin(val_sources_df["source_file"])].reset_index(drop=True)
    return {
        "train": meta_df[meta_df["source_file"].isin(set(train_sources_df["source_file"]))].reset_index(drop=True),
        "val": meta_df[meta_df["source_file"].isin(set(val_sources_df["source_file"]))].reset_index(drop=True),
        "test": meta_df[meta_df["source_file"].isin(set(test_sources_df["source_file"]))].reset_index(drop=True),
    }


In [ ]:
def read_audio_float32(path: Path) -> np.ndarray:
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if int(sr) != int(TARGET_SR):
        raise RuntimeError(f"SR tidak cocok: {sr} != {TARGET_SR} pada {path}")
    if audio.ndim > 1:
        audio = audio.mean(axis=1, dtype=np.float32)
    return np.ascontiguousarray(audio, dtype=np.float32)


def _resolve_clean_path(clean_path_text: str, preprocessed_dir: Path) -> Path:
    clean_path = Path(clean_path_text)
    if clean_path.exists():
        return clean_path
    candidate = preprocessed_dir / clean_path.name
    if candidate.exists():
        return candidate
    fail(f"File clean tidak ditemukan: {clean_path_text}")


def load_originals(preprocessed_dir: Path, n_samples: int) -> list[np.ndarray]:
    metadata_path = preprocessed_dir / "metadata.csv"
    if not metadata_path.exists():
        fail(f"metadata.csv tidak ditemukan: {metadata_path}")
    meta_df = pd.read_csv(metadata_path)
    for col in ["clean_path", "source_file"]:
        if col not in meta_df.columns:
            fail(f"metadata.csv tidak punya kolom {col}: {metadata_path}")
    splits = get_data_splits(meta_df)
    selected = _stratified_sample_table(splits["test"], min(int(n_samples), len(splits["test"])), seed=DATASET_RANDOM_SEED + 2)
    originals = []
    for _, row in selected.iterrows():
        originals.append(read_audio_float32(_resolve_clean_path(str(row["clean_path"]), preprocessed_dir)))
    return originals


def load_manifest(model_name: str, p: dict[str, Path]) -> pd.DataFrame:
    artifact_name = evaluation_artifact_name(model_name)
    manifest_path = p["outputs"] / artifact_name / "manifest.csv"
    if not manifest_path.exists():
        fail(f"Manifest rekonstruksi tidak ditemukan: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    required = {"gap_ms", "sample_index", "reconstructed_path"}
    missing = required - set(manifest.columns)
    if missing:
        fail(f"Manifest {manifest_path} tidak punya kolom: {sorted(missing)}")
    return manifest


def resolve_reconstructed_path(row: pd.Series, artifact_name: str, outputs_dir: Path) -> Path:
    recon_path = Path(str(row["reconstructed_path"]))
    if recon_path.exists():
        return recon_path
    fallback = (
        outputs_dir
        / artifact_name
        / f"gap_{int(row['gap_ms'])}ms"
        / f"{artifact_name}_gap{int(row['gap_ms'])}ms_sample{int(row['sample_index']):04d}_reconstructed.wav"
    )
    if fallback.exists():
        return fallback
    fail(f"File rekonstruksi tidak ditemukan. Manifest={recon_path} fallback={fallback}")


In [ ]:
def compute_lsd(original: np.ndarray, reconstructed: np.ndarray,
                sr: int = TARGET_SR, n_fft: int = 2048, hop_length: int = 512,
                gap_start: int | None = None, gap_end: int | None = None,
                frame_pad: int = 2) -> float:
    n = min(len(original), len(reconstructed))
    o = original[:n]
    r = reconstructed[:n]
    O = np.abs(librosa.stft(o, n_fft=n_fft, hop_length=hop_length)) ** 2
    R = np.abs(librosa.stft(r, n_fft=n_fft, hop_length=hop_length)) ** 2
    eps = max(1e-10, 1e-6 * float(O.max()))
    log_diff = 10.0 * (np.log10(O + eps) - np.log10(R + eps))
    if gap_start is not None and gap_end is not None:
        f_start = max(0, int(gap_start) // hop_length - frame_pad)
        f_end = min(O.shape[1], int(gap_end) // hop_length + frame_pad + 1)
        log_diff = log_diff[:, f_start:f_end]
    return float(np.mean(np.sqrt(np.mean(log_diff ** 2, axis=0))))


def _slice_metric_region(original, reconstructed, gap_start, gap_end):
    n = min(len(original), len(reconstructed))
    gap_start = max(0, min(int(gap_start), n))
    gap_end = max(gap_start, min(int(gap_end), n))
    ref = np.asarray(original[:n], dtype=np.float64)[gap_start:gap_end]
    est = np.asarray(reconstructed[:n], dtype=np.float64)[gap_start:gap_end]
    return ref, est


def compute_gap_snr(original, reconstructed, gap_start, gap_end):
    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)
    if len(ref) == 0:
        return np.nan
    noise = ref - est
    return float(10.0 * np.log10((np.sum(ref ** 2) + 1e-12) / (np.sum(noise ** 2) + 1e-12)))


def compute_gap_si_sdr(original, reconstructed, gap_start, gap_end):
    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)
    if len(ref) == 0:
        return np.nan
    ref = ref - np.mean(ref)
    est = est - np.mean(est)
    ref_energy = np.sum(ref ** 2) + 1e-12
    target = (np.sum(est * ref) / ref_energy) * ref
    error = est - target
    return float(10.0 * np.log10((np.sum(target ** 2) + 1e-12) / (np.sum(error ** 2) + 1e-12)))


def compute_gap_mel_distance(original, reconstructed, sr=TARGET_SR, gap_start=None, gap_end=None,
                             n_fft=1024, hop_length=256, n_mels=64):
    ref, est = _slice_metric_region(original, reconstructed, gap_start, gap_end)
    if len(ref) == 0:
        return np.nan
    if len(ref) < n_fft:
        pad = n_fft - len(ref)
        ref = np.pad(ref, (0, pad))
        est = np.pad(est, (0, pad))
    ref_mel = librosa.feature.melspectrogram(y=ref.astype(np.float32), sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=2.0)
    est_mel = librosa.feature.melspectrogram(y=est.astype(np.float32), sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=2.0)
    ref_peak = max(float(np.max(ref_mel)), 1e-10)
    ref_db = librosa.power_to_db(ref_mel, ref=ref_peak)
    est_db = librosa.power_to_db(est_mel, ref=ref_peak)
    frames = min(ref_db.shape[-1], est_db.shape[-1])
    return float(np.mean(np.abs(ref_db[..., :frames] - est_db[..., :frames])))


In [ ]:
EVAL_USE_FAD = os.environ.get("EVAL_USE_FAD", "1").lower() in {"1", "true", "yes", "on"}
EVAL_USE_GSTPEAQ = os.environ.get("EVAL_USE_GSTPEAQ", "1").lower() in {"1", "true", "yes", "on"}
EVAL_AUTO_INSTALL_DEPS = os.environ.get("EVAL_AUTO_INSTALL_DEPS", "1").lower() in {"1", "true", "yes", "on"}
FAD_USE_VGGISH_PCA = os.environ.get("FAD_USE_VGGISH_PCA", "0").lower() in {"1", "true", "yes", "on"}
FAD_DEBUG_STATS = os.environ.get("FAD_DEBUG_STATS", "1").lower() in {"1", "true", "yes", "on"}
GSTPEAQ_DIR = Path(os.environ.get("GSTPEAQ_DIR", project_root() / "external" / "gstpeaq")).resolve()
GSTPEAQ_BIN = os.environ.get("GSTPEAQ_BIN", "")
GSTPEAQ_PLUGIN = os.environ.get("GSTPEAQ_PLUGIN", "")
GSTPEAQ_ADVANCED = os.environ.get("GSTPEAQ_ADVANCED", "0").lower() in {"1", "true", "yes", "on"}
GSTPEAQ_AUTO_BUILD = os.environ.get("GSTPEAQ_AUTO_BUILD", "1").lower() in {"1", "true", "yes", "on"}
GSTPEAQ_REPO_URL = os.environ.get("GSTPEAQ_REPO_URL", "https://github.com/HSU-ANT/gstpeaq.git")
GSTPEAQ_REPO_BRANCH = os.environ.get("GSTPEAQ_REPO_BRANCH", "develop")


def _module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def _gstpeaq_binary_candidate() -> Path | None:
    candidates = [
        Path(GSTPEAQ_BIN) if GSTPEAQ_BIN else None,
        GSTPEAQ_DIR / "src" / "peaq",
        GSTPEAQ_DIR / "src" / "peaq.exe",
    ]
    for candidate in candidates:
        if candidate and candidate.exists():
            return candidate.resolve()
    return None


def _linux_command(command: list[str]) -> list[str]:
    import shutil

    if os.name != "nt" and hasattr(os, "geteuid") and os.geteuid() != 0 and shutil.which("sudo"):
        return ["sudo", *command]
    return command


def ensure_gstpeaq_backend() -> None:
    binary = _gstpeaq_binary_candidate()
    if binary is not None:
        info(f"GstPEAQ binary: {binary}")
        return
    if not GSTPEAQ_AUTO_BUILD:
        fail(
            "Executable GstPEAQ 'peaq' tidak ditemukan. Build external/gstpeaq terlebih dahulu "
            "atau set GSTPEAQ_BIN=/path/to/peaq. Ini tetap PEAQ_ODG yang sama seperti code_final_run_v2.py."
        )
    if os.name == "nt":
        fail(
            "Auto-build GstPEAQ hanya disediakan untuk Linux/Colab. Di Windows, build gstpeaq manual "
            "atau set GSTPEAQ_BIN ke peaq.exe."
        )

    info("GstPEAQ binary belum ada; menyiapkan build Linux/Colab untuk PEAQ_ODG resmi.")
    run_command(_linux_command(["apt-get", "update", "-qq"]))
    run_command(_linux_command([
        "apt-get", "install", "-y", "-qq",
        "git", "git2cl", "autoconf", "automake", "libtool", "pkg-config", "make", "gcc",
        "gtk-doc-tools", "w3-dtd-mathml",
        "libgstreamer1.0-dev", "libgstreamer-plugins-base1.0-dev",
        "gstreamer1.0-tools", "gstreamer1.0-plugins-base", "gstreamer1.0-plugins-good",
    ]))

    if not (GSTPEAQ_DIR / "autogen.sh").exists():
        GSTPEAQ_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_command([
            "git", "clone", "--depth", "1", "--branch", GSTPEAQ_REPO_BRANCH,
            GSTPEAQ_REPO_URL, str(GSTPEAQ_DIR),
        ])

    run_command(["bash", "autogen.sh"], cwd=GSTPEAQ_DIR)
    run_command(["make", "-j2"], cwd=GSTPEAQ_DIR)

    binary = _gstpeaq_binary_candidate()
    if binary is None:
        fail(f"Build GstPEAQ selesai tapi peaq tidak ditemukan di {GSTPEAQ_DIR / 'src' / 'peaq'}")
    info(f"GstPEAQ binary siap: {binary}")


def ensure_eval_metric_backends() -> None:
    if EVAL_USE_FAD and not _module_available("torchvggish"):
        if not EVAL_AUTO_INSTALL_DEPS:
            fail("torchvggish belum terinstall. Install `torchvggish` atau set EVAL_USE_FAD=0 untuk skip FAD.")
        info("torchvggish belum ada; installing backend FAD seperti requirements pipeline.")
        run_command([sys.executable, "-m", "pip", "install", "-q", "torchvggish"])
        importlib.invalidate_caches()
        if not _module_available("torchvggish"):
            fail("Install torchvggish selesai tetapi modul masih tidak bisa di-import. Restart runtime lalu run notebook dari awal.")
    if EVAL_USE_FAD and not _module_available("torch"):
        fail("PyTorch belum terinstall, padahal FAD VGGish membutuhkan torch. Aktifkan runtime Colab yang punya torch atau install torch dulu.")
    if EVAL_USE_GSTPEAQ:
        info(f"GstPEAQ enabled; binary search root: {GSTPEAQ_DIR}")
        ensure_gstpeaq_backend()


ensure_eval_metric_backends()


def extract_fad_features(audio_list: list[np.ndarray], sr: int = TARGET_SR) -> np.ndarray:
    features = []
    try:
        import inspect
        import torch
        import torchvggish
        try:
            from torchvggish import vggish_input
        except Exception:
            vggish_input = getattr(torchvggish, "vggish_input", None)

        vggish_kwargs = {}
        try:
            sig = inspect.signature(torchvggish.vggish)
            if "postprocess" in sig.parameters:
                vggish_kwargs["postprocess"] = bool(FAD_USE_VGGISH_PCA)
        except (TypeError, ValueError):
            pass

        device = torch.device("cpu")
        model = torchvggish.vggish(**vggish_kwargs).to(device).eval()
        postprocess_active = bool(vggish_kwargs.get("postprocess", False))
        postprocess_active = postprocess_active or bool(getattr(model, "postprocess", False))
        postprocess_active = postprocess_active or bool(getattr(model, "pproc", None) is not None)
        if postprocess_active and not FAD_USE_VGGISH_PCA:
            raise RuntimeError(
                "torchvggish tetap mengaktifkan VGGish post-processing walau FAD_USE_VGGISH_PCA=0. "
                "Gunakan torchvggish yang mendukung vggish(postprocess=False), atau set FAD_USE_VGGISH_PCA=1 "
                "kalau memang ingin skala PCA+8-bit legacy."
            )
        if vggish_input is None:
            raise RuntimeError("torchvggish.vggish_input tidak tersedia")

        with torch.inference_mode():
            for audio in audio_list:
                audio = np.asarray(audio, dtype=np.float32)
                if audio.ndim > 1:
                    audio = audio.mean(axis=1, dtype=np.float32)
                audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
                audio_16k = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                examples = vggish_input.waveform_to_examples(audio_16k, 16000)
                if len(examples) == 0:
                    raise RuntimeError("VGGish tidak menghasilkan example untuk salah satu audio.")
                examples = torch.as_tensor(examples, dtype=torch.float32, device=device)
                emb = model(examples).detach().cpu().numpy()
                if emb.dtype == np.uint8:
                    emb = emb.astype(np.float32)
                features.append(emb)
    except Exception as exc:
        raise RuntimeError(
            "FAD wajib memakai VGGish CPU. Install/konfigurasi torchvggish sebelum evaluasi FAD. "
            f"Detail: {exc}"
        ) from exc

    features = np.concatenate(features, axis=0).astype(np.float64, copy=False)
    if features.ndim != 2 or features.shape[0] < 2:
        raise RuntimeError(f"Embedding VGGish FAD tidak cukup: shape={features.shape}")
    if not np.isfinite(features).all():
        raise RuntimeError("Embedding VGGish FAD berisi NaN/Inf.")
    if FAD_DEBUG_STATS:
        info(
            "FAD VGGish features: "
            f"shape={features.shape}, mean={features.mean():.4f}, std={features.std():.4f}, "
            f"min={features.min():.4f}, max={features.max():.4f}"
        )
    return features


def compute_fad(original_audios: list[np.ndarray], reconstructed_audios: list[np.ndarray], sr: int = TARGET_SR) -> float:
    orig_features = extract_fad_features(original_audios, sr)
    recon_features = extract_fad_features(reconstructed_audios, sr)
    if orig_features.shape[1] != recon_features.shape[1]:
        raise RuntimeError(
            f"Dimensi embedding FAD tidak cocok: original={orig_features.shape}, recon={recon_features.shape}"
        )

    mu1 = np.mean(orig_features, axis=0)
    mu2 = np.mean(recon_features, axis=0)
    d = orig_features.shape[1]
    if len(orig_features) < 2 or len(recon_features) < 2:
        raise RuntimeError(
            f"FAD butuh minimal 2 embedding per set: original={len(orig_features)}, recon={len(recon_features)}"
        )

    sigma1 = np.cov(orig_features, rowvar=False) + 1e-6 * np.eye(d)
    sigma2 = np.cov(recon_features, rowvar=False) + 1e-6 * np.eye(d)
    sigma1 = (sigma1 + sigma1.T) * 0.5
    sigma2 = (sigma2 + sigma2.T) * 0.5

    diff = mu1 - mu2
    mean_diff = float(np.dot(diff, diff))
    if FAD_DEBUG_STATS:
        if len(orig_features) <= d or len(recon_features) <= d:
            info(
                "FAD sample count lebih kecil/sama dari dimensi embedding "
                f"(orig={len(orig_features)}, recon={len(recon_features)}, dim={d}); "
                "covariance FAD bisa noisy. Naikkan N_EVAL_SAMPLES untuk hasil final."
            )
        info(f"FAD mean term: {mean_diff:.4f}")

    def _psd_matrix_sqrt(mat: np.ndarray, eps: float = 1e-10) -> np.ndarray:
        mat = (mat + mat.T) * 0.5
        vals, vecs = np.linalg.eigh(mat)
        vals = np.clip(vals, eps, None)
        return (vecs * np.sqrt(vals)) @ vecs.T

    try:
        sqrt_sigma1 = _psd_matrix_sqrt(sigma1)
        covmean = _psd_matrix_sqrt(sqrt_sigma1 @ sigma2 @ sqrt_sigma1)
    except np.linalg.LinAlgError:
        offset = np.eye(d) * 1e-5
        sqrt_sigma1 = _psd_matrix_sqrt(sigma1 + offset)
        covmean = _psd_matrix_sqrt(sqrt_sigma1 @ (sigma2 + offset) @ sqrt_sigma1)

    fad = mean_diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    if not np.isfinite(fad):
        raise RuntimeError("FAD menghasilkan NaN/Inf.")
    return float(max(0.0, np.real(fad)))


def _find_gstpeaq_binary() -> str:
    import shutil

    candidates = [
        GSTPEAQ_BIN,
        str(GSTPEAQ_DIR / "src" / "peaq"),
        str(GSTPEAQ_DIR / "src" / "peaq.exe"),
        shutil.which("peaq"),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(Path(candidate).resolve())
    raise FileNotFoundError(
        "Executable GstPEAQ 'peaq' tidak ditemukan. Build external/gstpeaq terlebih dahulu "
        "atau set GSTPEAQ_BIN=/path/to/peaq. Set EVAL_USE_GSTPEAQ=0 hanya jika ingin skip PEAQ_ODG."
    )


def _find_gstpeaq_plugin() -> str | None:
    candidates = [
        GSTPEAQ_PLUGIN,
        str(GSTPEAQ_DIR / "src" / ".libs" / "libgstpeaq.so"),
        str(GSTPEAQ_DIR / "src" / ".libs" / "libgstpeaq.dylib"),
        str(GSTPEAQ_DIR / "src" / ".libs" / "gstpeaq.dll"),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(Path(candidate).resolve())
    return None


def _write_peaq_wav(path: Path, audio: np.ndarray, sr: int) -> None:
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=-1)
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000).astype(np.float32)
    sf.write(path, np.clip(audio, -1.0, 1.0), 48000, subtype="PCM_16")


def compute_gstpeaq_odg(original: np.ndarray, reconstructed: np.ndarray,
                        sr: int = TARGET_SR, temp_root: Path | None = None) -> float:
    import re
    import tempfile

    peaq_bin = _find_gstpeaq_binary()
    peaq_plugin = _find_gstpeaq_plugin()
    min_len = min(len(original), len(reconstructed))
    original = np.asarray(original[:min_len], dtype=np.float32)
    reconstructed = np.asarray(reconstructed[:min_len], dtype=np.float32)

    tmp_parent = Path(temp_root or project_root()).resolve()
    tmp_parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="gstpeaq_", dir=tmp_parent) as tmpdir:
        ref_path = Path(tmpdir) / "ref.wav"
        test_path = Path(tmpdir) / "test.wav"
        _write_peaq_wav(ref_path, original, sr)
        _write_peaq_wav(test_path, reconstructed, sr)

        cmd = [peaq_bin, "--gst-disable-segtrap"]
        if peaq_plugin:
            cmd.append(f"--gst-plugin-load={peaq_plugin}")
        if GSTPEAQ_ADVANCED:
            cmd.append("--advanced")
        cmd.extend([str(ref_path), str(test_path)])

        env = os.environ.copy()
        env["LC_ALL"] = "C"
        proc = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=120)
        output = f"{proc.stdout}\n{proc.stderr}"
        if proc.returncode != 0:
            raise RuntimeError(f"GstPEAQ gagal dengan exit code {proc.returncode}: {output.strip()}")

    match = re.search(r"Objective Difference Grade:\s*([-+]?\d+(?:\.\d+)?)", output)
    if not match:
        raise RuntimeError(f"Output GstPEAQ tidak memuat ODG: {output.strip()}")
    odg = float(match.group(1))
    if not np.isfinite(odg):
        raise RuntimeError("GstPEAQ menghasilkan ODG NaN/Inf.")
    return odg


compute_peaq_odg = compute_gstpeaq_odg
compute_peaq = compute_gstpeaq_odg





In [ ]:
def compute_results_from_outputs(model_name: str, p: dict[str, Path]) -> pd.DataFrame:
    manifest = load_manifest(model_name, p)
    artifact_name = evaluation_artifact_name(model_name)
    n_samples = int(manifest["sample_index"].astype(int).max()) + 1
    originals = load_originals(p["preprocessed"], n_samples)
    if len(originals) < n_samples:
        fail(f"Original samples kurang: butuh {n_samples}, dapat {len(originals)}")

    results = []
    for gap_ms in GAP_DURATIONS_MS:
        subset = manifest[manifest["gap_ms"].astype(int) == int(gap_ms)].sort_values("sample_index")
        if subset.empty:
            info(f"Skip gap {gap_ms}ms: tidak ada di manifest")
            continue
        info(f"Computing metrics {model_name} gap {gap_ms}ms ({len(subset)} files)")
        original_audios = []
        reconstructed_audios = []
        lsd_scores = []
        lsd_gap_only_scores = []
        gap_si_sdr_scores = []
        gap_snr_scores = []
        gap_mel_scores = []
        gstpeaq_odg_scores = []

        for _, row in subset.iterrows():
            sample_index = int(row["sample_index"])
            orig = originals[sample_index]
            recon = read_audio_float32(resolve_reconstructed_path(row, artifact_name, p["outputs"]))
            if "gap_start" in row and "gap_end" in row and pd.notna(row["gap_start"]) and pd.notna(row["gap_end"]):
                gap_start, gap_end = int(row["gap_start"]), int(row["gap_end"])
            else:
                _, gap_start, gap_end = make_eval_gap_mask(len(orig), int(gap_ms), sample_index=sample_index, sr=TARGET_SR)

            original_audios.append(orig)
            reconstructed_audios.append(recon)
            lsd_scores.append(compute_lsd(orig, recon, TARGET_SR, gap_start=gap_start, gap_end=gap_end, frame_pad=2))
            lsd_gap_only_scores.append(compute_lsd(orig, recon, TARGET_SR, gap_start=gap_start, gap_end=gap_end, frame_pad=0))
            gap_si_sdr_scores.append(compute_gap_si_sdr(orig, recon, gap_start, gap_end))
            gap_snr_scores.append(compute_gap_snr(orig, recon, gap_start, gap_end))
            gap_mel_scores.append(compute_gap_mel_distance(orig, recon, sr=TARGET_SR, gap_start=gap_start, gap_end=gap_end))

        if EVAL_USE_FAD:
            info(f"Computing FAD {model_name} gap {gap_ms}ms")
            fad_score = compute_fad(original_audios, reconstructed_audios, TARGET_SR)
        else:
            fad_score = np.nan

        if EVAL_USE_GSTPEAQ:
            info(f"Computing GstPEAQ ODG {model_name} gap {gap_ms}ms")
            for orig, recon in zip(original_audios, reconstructed_audios):
                gstpeaq_odg_scores.append(compute_gstpeaq_odg(orig, recon, TARGET_SR, temp_root=p["outputs"]))
        gstpeaq_odg = float(np.nanmean(gstpeaq_odg_scores)) if gstpeaq_odg_scores else np.nan

        results.append({
            "gap_ms": int(gap_ms),
            "gap_position": EVAL_GAP_POSITION,
            "LSD": round(float(np.mean(lsd_scores)), 4),
            "LSD_GAP_ONLY": round(float(np.mean(lsd_gap_only_scores)), 4),
            "GAP_LSD": round(float(np.mean(lsd_gap_only_scores)), 4),
            "GAP_SI_SDR": round(float(np.nanmean(gap_si_sdr_scores)), 4),
            "GAP_SNR": round(float(np.nanmean(gap_snr_scores)), 4),
            "GAP_MEL_DISTANCE": round(float(np.nanmean(gap_mel_scores)), 4),
            "FAD": round(float(fad_score), 4) if np.isfinite(fad_score) else np.nan,
            "VISQOL_ODG": np.nan,
            "PEAQ_ODG": round(float(gstpeaq_odg), 4) if np.isfinite(gstpeaq_odg) else np.nan,
            "GSTPEAQ_ODG": round(float(gstpeaq_odg), 4) if np.isfinite(gstpeaq_odg) else np.nan,
            "GAP_WINDOW_VISQOL_ODG": np.nan,
        })
    return pd.DataFrame(results)


def save_results_like_pipeline(results_df: pd.DataFrame, model_name: str, p: dict[str, Path]) -> None:
    p["results"].mkdir(parents=True, exist_ok=True)
    results_df = results_df.copy()
    artifact_name = evaluation_artifact_name(model_name)
    model_path = p["results"] / f"{artifact_name}_results.csv"
    optional_cols = ["FAD", "VISQOL_ODG", "PEAQ_ODG", "GSTPEAQ_ODG", "GAP_WINDOW_VISQOL_ODG"]
    if model_path.exists():
        old = pd.read_csv(model_path)
        if "gap_position" not in old.columns:
            old["gap_position"] = "center"
        old = old[old["gap_position"] == EVAL_GAP_POSITION]
        for col in optional_cols:
            if col in results_df.columns and col in old.columns:
                lookup = old.set_index("gap_ms")[col].to_dict()
                results_df[col] = results_df.apply(
                    lambda row: lookup.get(row["gap_ms"], row[col]) if pd.isna(row[col]) else row[col],
                    axis=1,
                )
    results_df["model"] = model_name
    results_df["experiment_config_id"] = EXPERIMENT_CONFIG_ID
    results_df["target_sr"] = TARGET_SR
    results_df["segment_samples"] = SEGMENT_SAMPLES
    if "gap_position" not in results_df.columns:
        results_df["gap_position"] = EVAL_GAP_POSITION
    results_df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    results_df.to_csv(model_path, index=False)
    info(f"Saved model results: {model_path}")

    master_path = p["results"] / "all_results.csv"
    if master_path.exists():
        existing = pd.read_csv(master_path)
        if "gap_position" not in existing.columns:
            existing["gap_position"] = "center"
        existing = existing[~((existing["model"] == model_name) & (existing["gap_position"] == EVAL_GAP_POSITION))]
        combined = pd.concat([existing, results_df], ignore_index=True)
    else:
        combined = results_df
    combined.to_csv(master_path, index=False)
    info(f"Updated master all_results: {master_path}")


def main() -> None:
    maybe_mount_google_drive()
    p = paths()
    info(f"model        : {MODEL_NAME}")
    info(f"stage root   : {p['stage']}")
    info(f"preprocessed : {p['preprocessed']}")
    info(f"outputs      : {p['outputs']}")
    info(f"results      : {p['results']}")
    results_df = compute_results_from_outputs(MODEL_NAME, p)
    print(results_df.to_string(index=False))
    save_results_like_pipeline(results_df, MODEL_NAME, p)
    info("Done.")


if __name__ == "__main__":
    main()


